# 0902 Classify Web Results
NAICS + OpenSecrets classification from web search summaries. Run after `0901_web_search.ipynb`.

In [27]:
import csv
from datetime import date
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError
from tqdm.notebook import tqdm

from config import COLUMNS, require_columns
from naics_data import ALL_NAICS as NAICS_DESCRIPTIONS

load_dotenv()

True

## Config

In [28]:
# Input: one or more web_search output CSVs
INPUT_PATHS = [
    #"09_outputs/web_search_batch_0_2026-08-12.csv",
    #"09_outputs/web_search_full_no_batch_2026-08-12.csv",
    #"09_outputs/web_search_from_cache_2026-08-13.csv"
    #"09_outputs/web_search_pac_contrib_batch_0_2026-08-13.csv"
    #"09_outputs/web_search_major_donors_missed_0_2026-08-16.csv", # major donors initially missed
    #"09_outputs/web_search_pac_contrib_batch_r2_0_2026-08-15.csv", # remaining pac contributos
    #"09_outputs/web_search_pac_contrib_batch_r2_1_2026-08-15.csv",
    #"09_outputs/web_search_pac_contrib_batch_r2_2_2026-08-16.csv",
    "09_outputs/web_search_contributors_combined_all_2026-08-16.csv"
]

PROVIDER    = "gemini"   # "gemini", "anthropic", or "openai"
DIGITS      = 2          # 2 = sector-level NAICS; 4 = industry group
BATCH_SIZE  = 50         # classification units per LLM call
TEST_N      = None         # rows to classify in test mode; set to None for full run
RANDOM_SEED = 42

PROVIDER_MODELS = {
    "anthropic": "claude-sonnet-4-6",
    "openai":    "gpt-4.1",
    "gemini":    "gemini-2.5-flash",
}

_NO_SUMMARY    = {"not employed", "unknown", "", "did not find", "none"}
_NO_OCCUPATION = {"none", "unknown", "not employed", "retired", "homemaker", "student"}
_UNINFORMATIVE = {"NONE", "UNKNOWN", "N/A", "NA", "NOT EMPLOYED", "UNEMPLOYED", "RETIRED", "HOMEMAKER", "STUDENT"}

## Load reference data

In [29]:
_DIR = Path(".")

def _load_os_categories(filename="open_secrets_level2_categories.csv") -> list[str]:
    path = _DIR / "09_inputs" / filename
    with open(path, newline="", encoding="utf-8") as f:
        return [row["x"] for row in csv.DictReader(f) if row["x"].strip()]

def _load_naics_reference(filename: str, key_col: str) -> dict[str, str]:
    path = _DIR / "09_inputs" / filename
    with open(path, newline="", encoding="utf-8") as f:
        return {row[key_col]: row["description"] for row in csv.DictReader(f)}

OS_CATEGORIES            = _load_os_categories()
NAICS_SECTOR_REFERENCE   = _load_naics_reference("naics_sector_title_expanded_with_custom_codes.csv", "naics_sector")
NAICS_INDUSTRY_REFERENCE = _load_naics_reference("naics_industry_title_expanded.csv", "naics_industry")

# '100' isn't a real NAICS code — add manually so it appears as a valid choice
#NAICS_SECTOR_REFERENCE["100"] = "Retired, Homemaker, Student, or Unemployed"

print(f"Loaded {len(OS_CATEGORIES)} OpenSecrets categories, "
      f"{len(NAICS_SECTOR_REFERENCE)} NAICS sectors, "
      f"{len(NAICS_INDUSTRY_REFERENCE)} NAICS industry groups")

Loaded 122 OpenSecrets categories, 21 NAICS sectors, 308 NAICS industry groups


## Helper functions

In [30]:
def _format_naics_industry_reference() -> str:
    lines = ["=== 2-DIGIT SECTORS ==="]
    for code, desc in sorted(NAICS_SECTOR_REFERENCE.items()):
        lines.append(f"  {code}: {desc}")
    lines.append("")
    lines.append("=== 4-DIGIT INDUSTRY GROUPS ===")
    current_sector = None
    for code in sorted(NAICS_INDUSTRY_REFERENCE):
        sector = code[:2]
        if sector != current_sector:
            current_sector = sector
            lines.append(f"  -- {sector}: {NAICS_SECTOR_REFERENCE.get(sector, '')} --")
        lines.append(f"  {code}: {NAICS_INDUSTRY_REFERENCE[code]}")
    return "\n".join(lines)


def _valid_codes_for(digits: int) -> dict[str, str]:
    return NAICS_INDUSTRY_REFERENCE if digits == 4 else NAICS_SECTOR_REFERENCE


def validate_naics(code: str, digits: int = 2) -> str:
    """Validate against reference CSVs, falling back to 2-digit sector or '00'."""
    code = str(code).strip()
    if code in _valid_codes_for(digits):
        return code
    if code in NAICS_SECTOR_REFERENCE:
        return code
    prefix = code[:2]
    if prefix in NAICS_SECTOR_REFERENCE:
        return prefix
    return "00"


def build_system_prompt(digits: int = 2) -> str:
    if digits == 4:
        code_label = "4-digit NAICS industry group code"
        code_list  = _format_naics_industry_reference()
    else:
        code_label = "2-digit NAICS sector code"
        code_list  = "\n".join(f"  {c}: {d}" for c, d in sorted(NAICS_SECTOR_REFERENCE.items()))

    os_list = "\n".join(f"  - {cat}" for cat in OS_CATEGORIES)

    return (
        "You are an expert at classifying businesses by NAICS industry code.\n"
        "Each entry below is a pre-researched industry summary for a California campaign finance contributor.\n"
        f"Assign each the single most appropriate {code_label} based solely on the information provided.\n\n"
        "Rules:\n"
        "- Always use a code from the valid list below.\n"
        "- For individuals: classify by the employer's industry\n"
        "- For self-employed / sole proprietors: classify by the occupation's industry\n"
        "- Any PAC or political committee, including those sponsored by unions, associations and political parties, should be classified as an '88' NAICS code.\n"
        "- When only an occupation is provided (no industry summary): classify based on that occupation's industry\n"
        "- Native American tribes and tribal governments should be classified as a '92a' NAICS code.\n"
        "- Use code '99' when no useful information is available to determine an industry.\n"
        '- naics_confidence: "high" if the summary is robust and clearly maps to one code; '
        '"medium" if classifying on occupation alone or summary is not specific about the organization\'s products or industry; '
        '"low" if the information is ambiguous, lacks detail, multi-sector, could match multiple NAICS codes or an exact search result was not found.\n'
        "- open_secrets_confidence: same logic as naics_confidence.\n"
        "- reasoning: 1 sentence citing the specific detail in the summary or occupation that drove the decision\n\n"
        f"VALID NAICS CODES:\n{code_list}"
        "- open_secrets_category: assign the single most appropriate OpenSecrets industry category from the list below\n"
        f"\nVALID OPENSECRETS CATEGORIES:\n{os_list}"
    )

## Models and schema

In [31]:
class NAICSResult(BaseModel):
    naics_code: str
    open_secrets_category: str
    naics_confidence: Literal["high", "medium", "low"]
    open_secrets_confidence: Literal["high", "medium", "low"]
    reasoning: str


class NAICSBatch(BaseModel):
    results: list[NAICSResult]


NOT_EMPLOYED_RESULT = NAICSResult(
    naics_code="100",
    open_secrets_category="Retired/Homemaker/Student/Unemployed",
    naics_confidence="high",
    open_secrets_confidence="high",
    reasoning="Contributor listed as not employed.",
)

NAICS_TOOL_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "naics_code":               {"type": "string"},
                    "open_secrets_category":    {"type": "string"},
                    "naics_confidence":         {"type": "string", "enum": ["high", "medium", "low"]},
                    "open_secrets_confidence":  {"type": "string", "enum": ["high", "medium", "low"]},
                    "reasoning":                {"type": "string"},
                },
                "required": ["naics_code", "open_secrets_category", "naics_confidence", "open_secrets_confidence", "reasoning"],
            }
        }
    },
    "required": ["results"],
}

## LLM provider functions

In [32]:
def make_client(provider: str):
    if provider == "gemini":
        from google import genai
        return genai.Client()
    if provider == "openai":
        import openai
        return openai.OpenAI()
    if provider == "anthropic":
        import anthropic
        return anthropic.Anthropic()
    raise ValueError(f"Unknown provider: {provider}")


def _call_gemini(client, user_prompt: str, system_prompt: str, model: str) -> list[NAICSResult]:
    from google.genai import types
    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            response_schema=NAICSBatch,
        ),
    )
    try:
        return NAICSBatch.model_validate_json(response.text).results
    except (ValidationError, Exception) as e:
        print(f"  Parse error: {e}")
        return []


def _call_openai(client, user_prompt: str, system_prompt: str, model: str) -> list[NAICSResult]:
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        response_format=NAICSBatch,
        temperature=0,
    )
    parsed = response.choices[0].message.parsed
    return parsed.results if parsed else []


def _call_anthropic(client, user_prompt: str, system_prompt: str, model: str) -> list[NAICSResult]:
    tool = {
        "name": "submit_classifications",
        "description": "Submit NAICS classifications for all provided contributors",
        "input_schema": NAICS_TOOL_SCHEMA,
    }
    response = client.messages.create(
        model=model,
        max_tokens=8096,
        system=system_prompt,
        tools=[tool],
        tool_choice={"type": "tool", "name": "submit_classifications"},
        messages=[{"role": "user", "content": user_prompt}],
    )
    for block in response.content:
        if block.type == "tool_use" and block.name == "submit_classifications":
            try:
                return NAICSBatch.model_validate(block.input).results
            except ValidationError as e:
                print(f"  Validation error: {e}")
    return []


_PROVIDER_FNS = {
    "gemini":    _call_gemini,
    "openai":    _call_openai,
    "anthropic": _call_anthropic,
}

client = make_client(PROVIDER)
print(f"{PROVIDER} client ready")

gemini client ready


## Classification function

In [33]:
def classify_from_summaries(
    client,
    summaries: list[str],
    occupations: list[str] | None = None,
    digits: int = 2,
    provider: str = "gemini",
    batch_size: int = 50,
) -> list[NAICSResult | None]:
    model         = PROVIDER_MODELS[provider]
    system_prompt = build_system_prompt(digits)
    call_fn       = _PROVIDER_FNS[provider]
    effective_batch = batch_size if digits == 2 else min(batch_size, 15)

    final: list[NAICSResult | None] = [None] * len(summaries)
    to_classify: list[tuple[int, str]] = []

    for i, summary in enumerate(summaries):
        raw_occ    = occupations[i] if occupations else None
        occupation = ("" if raw_occ is None or not isinstance(raw_occ, str) else raw_occ).strip()
        summary_clean = (summary or "").strip()

        if summary_clean.lower() in _NO_SUMMARY:
            if occupation and occupation.lower() not in _NO_OCCUPATION:
                to_classify.append((i, f"Occupation (no industry summary available): {occupation}"))
            else:
                final[i] = NOT_EMPLOYED_RESULT
        else:
            to_classify.append((i, f"Industry summary: {summary_clean}"))

    sep = "\n\n" + "=" * 60 + "\n\n"
    for start in tqdm(range(0, len(to_classify), effective_batch), desc=provider):
        chunk = to_classify[start: start + effective_batch]
        user_prompt = (
            f"Classify these {len(chunk)} contributors.\n\n"
            + ("=" * 60 + "\n\n")
            + sep.join(
                f"Contributor {i + 1}:\n{entry_text}"
                for i, (_, entry_text) in enumerate(chunk)
            )
            + "\n\n" + "=" * 60 + "\n\n"
            + f"Return exactly {len(chunk)} classifications in the 'results' array, in the same order."
        )
        try:
            results = call_fn(client, user_prompt, system_prompt, model)
        except Exception as e:
            print(f"  Chunk error at {start}: {e}")
            continue

        if len(results) != len(chunk):
            print(f"  Warning: expected {len(chunk)}, got {len(results)}")

        for (original_idx, _), result in zip(chunk, results):
            code = validate_naics(result.naics_code, digits)
            if code != result.naics_code:
                result = NAICSResult(
                    naics_code=code,
                    naics_confidence=result.naics_confidence,
                    open_secrets_category=result.open_secrets_category,
                    open_secrets_confidence=result.open_secrets_confidence,
                    reasoning=result.reasoning + f" (Code corrected from {result.naics_code}.)",
                )
            final[original_idx] = result

    return final


def _is_uninformative(value) -> bool:
    return str(value).strip().upper() in _UNINFORMATIVE

## Load and prepare data

In [34]:
frames = []
for path in INPUT_PATHS:
    frame = pd.read_csv(path)
    require_columns(frame, path)
    for col in ("industry_summary", "confidence"):
        if col not in frame.columns:
            raise KeyError(f"{path} is missing column {col!r} -- is this a web_search output file?")
    frames.append(frame)

data = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
print(f"{len(data):,} rows from {len(INPUT_PATHS)} file(s)")

# backfill prominence columns for older web_search outputs
if "is_prominent" not in data.columns:
    data["is_prominent"] = False
if "prominence_reason" not in data.columns:
    data["prominence_reason"] = ""

name_col        = COLUMNS["name"]
employer_col    = COLUMNS["employer"]
occ_col         = COLUMNS["occupation"]
entity_type_col = COLUMNS["entity_type"]
unit_col        = "search_key"   # one classification per unique search result

from standardization_helpers import NOT_EMPLOYED as _NOT_EMPLOYED_SET

for col in (employer_col, occ_col):
    data[col] = data[col].fillna("")

unique = data.drop_duplicates(subset=[unit_col]).reset_index(drop=True)
print(f"{len(unique):,} unique classification units")
unique[[name_col, employer_col, occ_col, "industry_summary", "confidence"]].head()

3,963 rows from 1 file(s)
3,963 unique classification units


,standardized_name,standardized_employer_name,processed_occupation,industry_summary,confidence
0,"PRATHIKANTI, SRIDHAR",ORCA BEHAVIORAL HEALTH,PHYSICIAN,ORCA Behavioral Health provides specialized me...,high
1,"HERNANDEZ, JOSE",ORANGE COUNTY,ATTORNEY,Orange County government provides a wide range...,high
2,"KIESEL, PAUL",KIESEL LAW LLP,ATTORNEY,Kiesel Law LLP is a personal injury law firm t...,high
3,"MCCARTHY, LOUISE",COMMUNITY CLINIC ASSOCIATION,PRESIDENT,The Community Clinic Association of Los Angele...,high
4,"MOLINA, JOSEPH",NOT EMPLOYED,RETIRED,"Healthcare executive and physician, formerly t...",high


## Select subset

In [35]:
if TEST_N is not None:
    n = min(TEST_N, len(unique))
    unique_to_run = unique.sample(n=n, random_state=RANDOM_SEED).reset_index(drop=True)
    data_to_run   = data[data[unit_col].isin(unique_to_run[unit_col])].reset_index(drop=True)
    print(f"TEST: classifying {len(unique_to_run):,} of {len(unique):,} unique units")
else:
    unique_to_run = unique
    data_to_run   = data
    print(f"FULL: classifying {len(unique_to_run):,} unique units")

def _is_not_employed_row(row) -> bool:
    if (row.get(entity_type_col, "") or "").strip() != "individual":
        return False
    e = (row[employer_col] or "").strip().lower()
    o = (row[occ_col] or "").strip().lower()
    return (not e or e in _NOT_EMPLOYED_SET) and (not o or o in _NOT_EMPLOYED_SET)

not_employed_mask = unique_to_run.apply(_is_not_employed_row, axis=1)

to_classify  = unique_to_run[~not_employed_mask].reset_index(drop=True)
auto_retired = unique_to_run[not_employed_mask].reset_index(drop=True)
print(f"  {len(to_classify):,} to classify via LLM")
print(f"  {len(auto_retired):,} auto-assigned code 100 (not employed)")

to_classify[[name_col, employer_col, occ_col, "industry_summary"]].head()

FULL: classifying 3,963 unique units
  3,504 to classify via LLM
  459 auto-assigned code 100 (not employed)


,standardized_name,standardized_employer_name,processed_occupation,industry_summary
0,"PRATHIKANTI, SRIDHAR",ORCA BEHAVIORAL HEALTH,PHYSICIAN,ORCA Behavioral Health provides specialized me...
1,"HERNANDEZ, JOSE",ORANGE COUNTY,ATTORNEY,Orange County government provides a wide range...
2,"KIESEL, PAUL",KIESEL LAW LLP,ATTORNEY,Kiesel Law LLP is a personal injury law firm t...
3,"MCCARTHY, LOUISE",COMMUNITY CLINIC ASSOCIATION,PRESIDENT,The Community Clinic Association of Los Angele...
4,"DAKE, GLEN","GDML HOLDINGS, INC",ARCHITECT,"Glen Dake is a landscape architect, and GDML H..."


## Run classification

In [36]:
results = classify_from_summaries(
    client,
    to_classify["industry_summary"].fillna("").tolist(),
    occupations=to_classify[occ_col].fillna("").tolist(),
    digits=DIGITS,
    provider=PROVIDER,
    batch_size=BATCH_SIZE,
)
print(f"Classified {sum(r is not None for r in results)}/{len(results)} unique units")

gemini:   0%|          | 0/70 [00:00<?, ?it/s]

Classified 3504/3504 unique units


## Map results back to data

In [37]:
result_map = {}
for uid, result in zip(to_classify[unit_col], results):
    result_map[uid] = result
for uid in auto_retired[unit_col]:
    result_map[uid] = NOT_EMPLOYED_RESULT

def _lookup(uid, field):
    r = result_map.get(uid)
    return getattr(r, field, "") if r else ""

data_to_run["naics_code_llm"]          = data_to_run[unit_col].map(lambda u: _lookup(u, "naics_code"))
data_to_run["open_secrets_category"]   = data_to_run[unit_col].map(lambda u: _lookup(u, "open_secrets_category"))
data_to_run["naics_confidence"]        = data_to_run[unit_col].map(lambda u: _lookup(u, "naics_confidence"))
data_to_run["open_secrets_confidence"] = data_to_run[unit_col].map(lambda u: _lookup(u, "open_secrets_confidence"))
data_to_run["naics_reasoning"]         = data_to_run[unit_col].map(lambda u: _lookup(u, "reasoning"))
data_to_run["naics_description"]       = data_to_run["naics_code_llm"].map(lambda c: NAICS_DESCRIPTIONS.get(c, ""))

## Inspect results

In [38]:
inspect_cols = [
    name_col, employer_col, occ_col,
    "industry_summary", "confidence",
    "naics_code_llm", "naics_description", "naics_confidence",
    "open_secrets_category", "open_secrets_confidence",
]
data_to_run[[c for c in inspect_cols if c in data_to_run.columns]].head(20)

,standardized_name,standardized_employer_name,processed_occupation,industry_summary,confidence,naics_code_llm,naics_description,naics_confidence,open_secrets_category,open_secrets_confidence
0,"PRATHIKANTI, SRIDHAR",ORCA BEHAVIORAL HEALTH,PHYSICIAN,ORCA Behavioral Health provides specialized me...,high,60,Healthcare/Education,high,Health Services,high
1,"HERNANDEZ, JOSE",ORANGE COUNTY,ATTORNEY,Orange County government provides a wide range...,high,92,Government/Public Safety,high,Civil Servants/Public Officials,high
2,"KIESEL, PAUL",KIESEL LAW LLP,ATTORNEY,Kiesel Law LLP is a personal injury law firm t...,high,50,Professional/Legal,high,Lawyers & Lobbyists,high
3,"MCCARTHY, LOUISE",COMMUNITY CLINIC ASSOCIATION,PRESIDENT,The Community Clinic Association of Los Angele...,high,76,Associations,high,Business Associations,high
4,"MOLINA, JOSEPH",NOT EMPLOYED,RETIRED,"Healthcare executive and physician, formerly t...",high,100,Retired/Homemaker/Student/Unemployed,high,Retired/Homemaker/Student/Unemployed,high
5,"AYALA, RAUL",NOT EMPLOYED,NONE,Raul Ayala is a Federal Public Defender.,high,100,Retired/Homemaker/Student/Unemployed,high,Retired/Homemaker/Student/Unemployed,high
6,"DAKE, GLEN","GDML HOLDINGS, INC",ARCHITECT,"Glen Dake is a landscape architect, and GDML H...",high,50,Professional/Legal,high,Business Services,high
7,"COOPER, JOSH",AMERICAN COLLEGE OF RADIOLOGY,EXECUTIVE,The American College of Radiology is a profess...,high,76,Associations,high,Business Associations,high
8,"SAADIAN, BOBBY",WILSHIRE LAW FIRM,ATTORNEY,Wilshire Law Firm is a nationally recognized l...,high,50,Professional/Legal,high,Lawyers & Lobbyists,high
9,"SAFRAN, THOMAS",THOMAS SAFRAN AND ASSOCIATES,CHAIR,Thomas Safran & Associates is a multifamily ho...,high,40,Real Estate/Construction,high,Real Estate,high


## Save output

In [23]:
data_to_run.columns

Index(['rowid', 'Transaction.Type', 'Cycle', 'Election', 'Start.Date',
       'End.Date', 'Amount', 'Recipient.Name', 'Recipient.Committee',
       'Recipient.Committee.ID', 'Office', 'District', 'Ballot.Measure',
       'Contributor.Name', 'Contributor.ID', 'Contributor.City',
       'Contributor.State', 'Contributor.Zip.Code', 'Contributor.Employer',
       'Contributor.Occupation', 'Candidate.Contribution',
       'Ballot.Measure.Contribution', 'Allied.Committee', 'Normalized',
       'standardized_name', 'processed_name', 'standardized_employer_name',
       'processed_employer_name', 'standardized_occupation',
       'processed_occupation', 'standardized_city', 'zip_code_processed',
       'has_pac_language', 'entity_type', 'race_prop', 'row_id',
       'contribution_id', 'entity_id', 'race_total_amount',
       'crosses_threshold', 'cont_row_id', 'has_contributor_id', 'name_v1',
       'name_v2', 'name_v4', 'name_v5', 'needs_review', 'legacy_match_id',
       'uuid', 'FILER_ID', 

In [43]:
tag     = date.today().isoformat()
mode    = f"test{TEST_N}" if TEST_N is not None else "full"
out_dir = Path("09_outputs")
out_dir.mkdir(exist_ok=True)

slim_cols = [
    unit_col, COLUMNS["entity_id"], name_col, employer_col, occ_col,
    "industry_summary", "confidence", "is_prominent", "prominence_reason",
    "naics_code_llm", "naics_description", "open_secrets_category",
    "naics_confidence", "open_secrets_confidence", "naics_reasoning",
]
slim_cols = [c for c in slim_cols if c in data_to_run.columns]

# CHANGE NAME!
slim_path = out_dir / f"classification_inpute_combined_{mode}_{tag}.csv"
full_path = out_dir / f"classification_inpute_combined_{mode}_{tag}.csv"
data_to_run[slim_cols].to_csv(slim_path, index=False)
data_to_run.to_csv(full_path, index=False)
print(f"Saved slim: {slim_path}")
print(f"Saved full: {full_path}")

Saved slim: 09_outputs/classification_inpute_combined_full_2026-08-16.csv
Saved full: 09_outputs/classification_inpute_combined_full_2026-08-16.csv


## Expand to full uuid coverage

`data_to_run` above is deduped to one row per `search_key` (0901 drops all but
one representative `uuid` per unit before anything gets searched). Broadcast
this run's classification back onto *every* raw contribution row in
`classification_input.csv` that shares each unit's `search_key`, so the output
has one row per `uuid` -- matching what `assign_final_classification.Rmd`
expects for its `uuid`-based join against `rule`/`os`.

In [44]:
from expand_classifications import expand_to_full_coverage

expanded = expand_to_full_coverage(
    [str(full_path)],
    classification_input_path="../08_alternative_pipeline/08_outputs/classification_input_combined.csv"
)
expanded_path = out_dir / f"classification_input_combined_expanded_{mode}_{tag}.csv"
expanded.to_csv(expanded_path, index=False)
print(f"Saved expanded (one row per uuid): {expanded_path}")

  Expanded 1 file(s) covering 3,963 units -> 36,744 raw rows (36,744 classified, 0 still unclassified)
Saved expanded (one row per uuid): 09_outputs/classification_input_combined_expanded_full_2026-08-16.csv
